In [1]:
import pandas as pd
from dh_tool.excel.utils import get_cell_addresses
from dh_tool.excel import ExcelCore

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Score": [85, 92, 78],
    "Grade": ["A", "A", "B"]
})

# get_cell_addresses(df, )
condition = df["Score"] > 80
two_condition = df.map(lambda x: isinstance(x, str))
cells = get_cell_addresses(df, condition)
two_cells = get_cell_addresses(df, two_condition)
display(df)
print(cells, two_cells)

,Name,Score,Grade
0,Alice,85,A
1,Bob,92,A
2,Charlie,78,B


['B2', 'B3'] ['A2', 'C2', 'A3', 'C3', 'A4', 'C4']


In [ ]:
with ExcelCore('asdf.xlsx') as ec:
    ec.write( df,sheet_name='qwer',  style_preset=False)
    ec.style_to_cells(
        cells=two_cells,
        font={'name': 'Arial', 'size': 12, 'color': 'FF0000'},
        color='Blue'
    )
    ec.style(
        freeze_panes='A3',
    )
    ec.write(df, style_preset=False)
    # ec.write()
    ec.save()
import os
# os.unlink('asdf.xlsx')

ExcelCore opened.
Transaction committed.
Transaction committed.


In [ ]:
## freeze panes

In [ ]:
with ExcelCore('with_style_test.xlsx') as excel:
    (
        excel
        .write('test', df)
        .style(
            freeze_first_row=True,
            color='red',
            font = {'size': 42}
        )
        # .save()
        .write('condition', df)
        .style_to_cells(
            cells=cells,
            freeze_first_row=True,
            color='red',
            font = {'size': 42}
        )
        .style_to_cells(
            cells=two_cells,
            color='blue',
            font={
                "italic": True,
                "size": 8,
                "bold": True
            }
        )
        .save()
        .end()
    )

In [ ]:
cells

In [ ]:
df

In [ ]:
str_condition = df.applymap(lambda x: isinstance(x, str))
str_condition
get_cell_addresses(df, str_condition)

In [ ]:
str_condition

In [ ]:
condition.to_frame()

---

In [ ]:
from dh_tool.log_tool.logger import Logger 
from dh_tool.log_tool.handlers.console_handler import get_console_handler
from dh_tool.log_tool.decorators import auto_logger
from dh_tool.log_tool.context_managers import log_block
logger = Logger("name")
logger.add_handler(get_console_handler())

@auto_logger(logger)
def add(a, b):
    return a + b

with log_block(logger, "Computation Block"):
    result = add(5, 10)
    logger.info(f"Final Result: {result}")

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import os
api_key = os.getenv("OENAI_API_KEY")

In [ ]:
api_key

In [ ]:
from dh_tool import load, save
from dh_tool.common import *
from dh_tool.llm_tool import LLMConfig, GPTModel
from dh_tool.log_tool import Logger, get_console_handler, auto_logger, log_async_block

logger = Logger("my_logger", level="DEBUG")
logger.add_handler(get_console_handler())
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
logger.info(f"API key: {api_key}")

config = LLMConfig(
    model="gpt-4o-mini",
    api_key=api_key,
    generation_params={
        "max_completion_tokens": 12,
        "hi": 10,
        "temperature": 0.9,
    },
)

gpt = GPTModel(config)


# async with log_async_block(logger, "Generating gpt response"):
#     await gpt.generate("웃긴 얘기좀", parsed=True)
@auto_logger(logger)
async def test_gpt(text):
    result = await gpt.generate(text, parsed=True)
    return result

await test_gpt("웃긴 얘기좀")

In [ ]:
from dh_tool import ExcelCore
from dh_tool import load
from dh_tool.excel.style import MY_COLOR_MAP
from dh_tool.excel.utils import get_cell_addresses, generate_color_variants, get_full_column_ranges, get_full_row_cells

df = load('합불_모델별_구간별_결과.xlsx')

In [ ]:
variants = generate_color_variants(MY_COLOR_MAP['deep_purple']['hex'], 10)
# variants

In [ ]:

idx = 0

with ExcelCore('test.xlsx') as excel:
    excel.write('test', df)
    excel.style(
        freeze_first_row=True,
        filter=True,
    )
    for cut in df['구간'].unique():
        condition = df['구간'] == cut
        sub_df = df[condition]
        cells = get_cell_addresses(sub_df, condition)
        # print(cells)
        # full_column_cells = get_full_column_ranges(cells, len(sub_df))
        full_row_cells = get_full_row_cells(cells, sub_df.columns.tolist(), ['id', 'model'])
        # print(full_column_cells, )
        # print(full_row_cells)
        excel.style_to_cells(
            cells=full_row_cells,
            color=variants[idx],
            font={'size': 12}
        )
        idx += 1
        idx = idx % len(variants)
        # break
    excel.save()

In [ ]:
sub_df.columns

---

In [ ]:
from dh_tool.common import *
from dh_tool.llm_tool import LLMConfig, GPTModel, GeminiModel
from dh_tool.llm_tool.utils.stream_processor import GPTStreamProcessor, GeminiStreamProcessor

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
gpt = GPTModel(
    LLMConfig(
        model="gpt-4o-mini",
        api_key=api_key,
    ))
gemini_key = os.getenv("GEMINI_API_KEY")
gemini = GeminiModel(
    LLMConfig(
        model='gemini-1.5-flash', 
        api_key=gemini_key
    )
)

In [ ]:
stream = await gemini.generate_stream("안녕?")
stream_ret = await GeminiStreamProcessor.process_stream(stream, verbose=True)


In [ ]:
stream_ret

In [ ]:
stream.to_dict()

In [ ]:
from google.generativeai.types.generation_types import GenerateContentResponse 


In [ ]:
stream

In [ ]:
GenerateContentResponse.from_response(stream)